# Project B - Context-Gap Distillation

KL between the model's predictions with and without the skill document in context, used
as both the importance signal and the distillation loss.

More session-choppable than Project A: each skill-category adapter is self-contained and
`distill/train.py --resume` skips groups already trained. Budget ~5 GPU-hours.

In [ ]:
!pip -q install peft accelerate
%cd /kaggle/working
import os
REPO = '/kaggle/working/myrios'
if not os.path.exists(REPO):
    !git clone -q https://github.com/USER/myrios.git {REPO}
%cd {REPO}

RUNS = '/kaggle/working/artifacts/runs_skill'
SCORES = f'{RUNS}/scores_span.jsonl'

In [ ]:
ARCHIVE = '/kaggle/input/myrios-runs-skill/runs_skill.zip'
!python scripts/kaggle_sync.py restore --archive {ARCHIVE} --run-root {RUNS}
!python skills/generate_toy_skills.py

## Gate verification

The riskiest part of the pipeline. High-KL spans must be API identifiers, error codes and
rule clauses; low-KL spans must be markdown scaffolding. Inspect this by eye before
trusting anything trained on it.

In [ ]:
!python kl_gate/score.py --config configs/skill_base.yaml --granularity span --out {SCORES}
!python kl_gate/inspect_gate.py --scores {SCORES} --top-frac 0.25 --show 6 \
    --out {RUNS}/gate_report.json

## Distillation and evaluation

`random` is the control that matters: same active-token budget as the KL gate, spans
chosen at random. Any gap between it and `kl_top` is the gate doing real work.

In [ ]:
!python scripts/run_skills.py --config configs/skill_base.yaml --stages distill,eval,report
!python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs_skill.zip

In [ ]:
!python scripts/sweep_kl.py --config configs/skill_base.yaml --fracs 0.1,0.25,0.5 \
    --granularities span,token
!python eval/skill_report.py --report-dir {RUNS}/report --run-root {RUNS} \
    --out docs/results_skills.md

In [ ]:
from IPython.display import Image, display
display(Image(f'{RUNS}/report/figures/headline_skills.png'))
display(Image(f'{RUNS}/report/figures/kl_threshold_sweep.png'))
!python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs_skill.zip